<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">


# The Data Scientist
## Book 3 · Statistics, Machine Learning, and Model Evaluation
### Notebook 06 · Module 3 Evaluation Report

© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://thedatascientist.dev | https://linktr.ee/dyjh


### Why this notebook exists
The chapter asks for an evaluation report, not a model dump. The notebook
should read like a small project write-up that a colleague could follow.

This cell continues the worked example for the current section.


This cell prepares the notebook for local or Colab execution. It finds the book root, clones the companion repo when needed, and makes the working directory predictable.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "3_data"  # book folder to locate locally or after clone
REPO_URL = "https://github.com/yhilpisch/tdscode"  # companion repo with notebooks, data, and code

cwd = Path.cwd().resolve()  # start from the current working directory
BOOK_ROOT = None  # filled once a matching book directory is found
for candidate in [cwd, *cwd.parents]:  # search upward for the book root
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():  # local book root
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():  # repo/book layout
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")  # Colab clone location
    if not repo_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_root)], check=True)  # fetch the companion repo once
    BOOK_ROOT = repo_root / BOOK_NAME  # book folder inside the clone

os.chdir(BOOK_ROOT)  # keep relative paths anchored on the book root
if str(BOOK_ROOT) not in sys.path:  # allow src/ imports
    sys.path.insert(0, str(BOOK_ROOT))  # allow src/ imports

code_dir = BOOK_ROOT / "code"  # helper scripts live here
if code_dir.exists() and str(code_dir) not in sys.path:  # make helper scripts importable
    sys.path.insert(0, str(code_dir))

requirements = BOOK_ROOT / "requirements.txt"  # install only when present
if requirements.exists() and "google.colab" in sys.modules:  # keep Colab in sync with the book
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)  # keep Colab in sync with the book

print("Book root:", BOOK_ROOT)


In [ ]:
from pathlib import Path
import importlib.util

import pandas as pd

module_path = BOOK_ROOT / 'code' / 'churn_report.py'
spec = importlib.util.spec_from_file_location('churn_report', module_path)
assert spec is not None and spec.loader is not None
churn_report = importlib.util.module_from_spec(spec)
spec.loader.exec_module(churn_report)

print(f'Loaded {module_path.resolve()}')


This cell continues the worked example for the current section.


In [ ]:
df = churn_report.load_customers()

print(df.head().to_string(index=False))
print()
print('churn counts:')
print(df[churn_report.TARGET_COLUMN].value_counts().sort_index())
print()
print('numeric summary:')
print(df[churn_report.NUMERIC_COLUMNS].describe().round(2).to_string())


This cell fits the model on the training split and checks the held-out rows.


In [ ]:
X_train, X_valid, X_test, y_train, y_valid, y_test = churn_report.split_customers(df)
models = churn_report.build_models()

rows = []
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

    valid_pred = model.predict(X_valid)
    test_pred = model.predict(X_test)

    valid_scores = churn_report.evaluate_predictions(y_valid, valid_pred)
    test_scores = churn_report.evaluate_predictions(y_test, test_pred)

    rows.append({'model': name, 'split': 'validation', **valid_scores})
    rows.append({'model': name, 'split': 'test', **test_scores})

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))


This cell continues the worked example for the current section.


In [ ]:
validation_rows = comparison[comparison['split'] == 'validation'].copy()
best_name = validation_rows.sort_values(['recall', 'accuracy'], ascending=False).iloc[0]['model']
print('best validation model =', best_name)

best_model = fitted_models[best_name]
best_model.fit(pd.concat([X_train, X_valid]), pd.concat([y_train, y_valid]))

mistakes = churn_report.misclassified_customers(best_model, df)
print()
print(mistakes.head().to_string(index=False))


### Report Summary
The model choice should be explained in plain language. A good summary
says what was tried, which model was chosen, what it still gets wrong, and
what data would help next.

### Where We Are Heading Next
The module ends here. The next step in a real project would be to package
the notebook, the data, and the short README into a small public repo.

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">
